# F1 Race Intelligence — Exploratory Analysis & Verification
### Master's Degree in Data Science | University of Milan (UNIMI)
**Course:** Coding for Data Science and Data Management — Python Module  
**Project:** Box, Box — Formula 1 Race Analytics

This notebook provides a complete interactive walk-through of the project's analytical pipeline, importing and verifying each module in `src/f1_analysis/`.

## 1. Setup and Environment Verification

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.f1_analysis.config import RAW_DATA_DIR, JOLPICA_BASE_URL
from src.f1_analysis.data.api_client import F1ApiClient
from src.f1_analysis.data.data_loader import F1DataLoader
from src.f1_analysis.data.preprocessing import clean_lap_times, apply_fuel_correction, merge_laps_with_stints
from src.f1_analysis.analysis.driver_performance import compute_driver_pace_metrics, compare_teammates
from src.f1_analysis.analysis.tyre_analysis import compute_compound_degradation, compute_driver_stint_summary
from src.f1_analysis.analysis.pitstop_analysis import compute_pit_stop_summary
from src.f1_analysis.analysis.race_dynamics import compute_position_progression, compute_net_position_changes, compute_gap_to_leader
from src.f1_analysis.analysis.team_performance import compute_constructor_pace_ranking, compute_constructor_points_summary
from src.f1_analysis.visualization.charts import (
    plot_driver_pace_distributions,
    plot_tyre_degradation_regression,
    plot_position_progression_chart,
    create_interactive_gap_chart,
)

print("All modules imported successfully!")
print("Cache directory:", RAW_DATA_DIR)

## 2. Ingesting Real Race Data via F1DataLoader
We select the **2024 Bahrain Grand Prix (Round 1)** as our baseline dataset.

In [ ]:
loader = F1DataLoader()

# Load Race Results, Laps, and Pit Stops
df_results = loader.load_race_results(year=2024, round_num=1)
df_laps_raw = loader.load_race_laps(year=2024, round_num=1)
df_pitstops = loader.load_pit_stops(year=2024, round_num=1)

print(f"Loaded {len(df_results)} drivers in classification.")
print(f"Loaded {len(df_laps_raw)} raw lap records.")
print(f"Loaded {len(df_pitstops)} pit stops.")
df_results.head()

## 3. Data Cleaning and Fuel Load Correction
We filter out non-representative laps (in-laps, out-laps, safety cars) and apply the physics-based fuel mass correction (~0.035 s / lap).

In [ ]:
df_clean = clean_lap_times(df_laps_raw)
df_laps = apply_fuel_correction(df_clean)

valid_count = df_laps['is_valid_representative_lap'].sum()
total_count = len(df_laps)
print(f"Clean representative laps: {valid_count} / {total_count} ({valid_count/total_count:.1%})")
df_laps[['lap_number', 'driver_id', 'lap_time_sec', 'fuel_corrected_lap_sec', 'is_valid_representative_lap']].head()

## 4. Module A: Driver Pace & Statistical Consistency

In [ ]:
# Compute pace metrics (Median, Mean, Standard Deviation, IQR)
driver_pace = compute_driver_pace_metrics(df_laps, use_fuel_corrected=True)
display(driver_pace.head(10))

# Visualize Pace Distribution Boxplot
fig = plot_driver_pace_distributions(df_laps, top_n=10, use_fuel_corrected=True)
plt.show()

### Teammate Pace Hypothesis Testing

In [ ]:
teammates = compare_teammates(df_laps, df_results, use_fuel_corrected=True)
display(teammates)

## 5. Module B: Tyre Degradation & OLS Regression

In [ ]:
# Synthesize compound stint mapping for demonstration
df_laps_stints = merge_laps_with_stints(df_laps, pd.DataFrame())
stint_summary = compute_driver_stint_summary(df_laps_stints, use_fuel_corrected=True)
display(stint_summary.head(10))

## 6. Module C: Pit Stop Strategy

In [ ]:
pit_summary = compute_pit_stop_summary(df_pitstops, df_results)
display(pit_summary.head(10))

## 7. Module D: Race Dynamics & Lap Progression Chart

In [ ]:
# Broadcast lap chart
fig_pos = plot_position_progression_chart(df_laps, top_n=10)
plt.show()

# Net Positions Gained/Lost
net_positions = compute_net_position_changes(df_results)
display(net_positions.head(10))

## 8. Module E: Constructor Performance

In [ ]:
team_pace = compute_constructor_pace_ranking(df_laps, df_results, use_fuel_corrected=True)
display(team_pace)

team_points = compute_constructor_points_summary(df_results)
display(team_points)